# 課題 3 ヒント

課題 3 (= LLM workflow) の `refund_agent` 内 L98 / L104 では、 **ワークショップ本編で明示的には扱わなかった OpenAI ChatCompletion API の tool calling** を直接実装します (= 04 までは LangChain `create_agent` 等の上位 abstraction で隠れていた部分)。 そのため自力で進めるのが難しい所があるかもしれないので、 以下にヒントを用意しました。

各セクションは折り畳まれているので、 まず自力で挑戦して、 詰まった時に **必要なものだけクリックで展開** してください。

<details>
<summary>★ 実装の流れ (L98 + L104 全体) クリックで展開</summary>

`refund_agent` の MCP + LLM tool calling の全体フロー:

1. **ツール一覧取得** — MCP server から利用可能な tool list を取る
2. **OpenAI 形式に変換** — MCP の Tool object を OpenAI の `tools` 引数の形式に組み替える
3. **LLM 呼出** — 変換した tool list を LLM に渡して 「tool 使うべきか + どう使うか」 を判断させる
4. **判定** — response の `finish_reason` (= 何で止まったか) を見て、 tool 使うべきだと判断したか check
5. **tool 呼出** — LLM の指示通り MCP server に対して tool を実行
6. **結果を LLM へ返却** — tool 結果を **どの tool call の応答か紐付けて** (= `tool_call_id`) message に format
7. (= 既存コード L107 以降が tool_message と結果を使って user 向け content を組み立てる)

→ 1-3 が L98、 4-6 が L104 の作業範囲。

</details>

<details>
<summary>★ ヒント 1: MCP <code>list_tools</code> の戻り値の形 (クリックで展開)</summary>

`await mcp_client.session.list_tools()` のレスポンス構造:

```
ListToolsResult
  tools: [
    Tool(name="invoice_lookup", description="...", inputSchema={"type":"object", ...}),
    Tool(name="invoice_refund", description="...", inputSchema={"type":"object", ...}),
  ]
```

</details>

<details>
<summary>★ ヒント 2: OpenAI が要求する <code>tools</code> 引数の format (クリックで展開)</summary>

`chat.completions.create(tools=...)` に渡す形:

```python
[
    {
        "type": "function",
        "function": {
            "name": "invoice_lookup",
            "description": "...",
            "parameters": {"type": "object", ...},   # ← MCP の inputSchema をここに
        }
    },
    ...
]
```

</details>

<details>
<summary>★ ヒント 3: <code>chat.completions.create</code> の response 構造 (クリックで展開)</summary>

`response` 変数の中身:

```
ChatCompletion
  choices: [
    Choice
      finish_reason: "tool_calls"  or  "stop"
      message: ChatCompletionMessage
        content: None or "text"
        tool_calls: [
          ChatCompletionMessageToolCall
            id: "call_xyz"          ← 後段の tool_call_id に必要
            function:
              name: "invoice_lookup"
              arguments: '{"customer_first_name":"...", ...}'   ← JSON 文字列、 dict ではない
        ]
  ]
```

</details>

<details>
<summary>★ ヒント 4: tool 結果メッセージの format (クリックで展開)</summary>

L104 で構築する `tool_message` 変数の形 (= 既存コード L111 / L120 / L127 が参照):

```python
{
    "role": "tool",
    "tool_call_id": ...,   # ← 上の tool_call の id 値
    "name": ...,           # tool 名
    "content": ...,        # tool 実行結果の文字列
}
```

OpenAI spec で **`tool_call_id` 必須**、 LLM が後続 turn で 「どの tool call の結果か」 を紐付けるため。

</details>

<details>
<summary>★ ヒント 5: StateGraph 配線 (L187) — <code>add_conditional_edges</code> は要らない (クリックで展開)</summary>

`intent_classifier` 関数の中身を見ると、 内部で **`Command(goto=...)` を返してる**:

```python
elif response.intent == "QNA":
    return Command(goto="qna_agent")
elif response.intent == "REFUND":
    return Command(goto="refund_agent")
```

これは **動的 routing を node 内で完結** させる pattern (= LangGraph 1.x 流)。 つまり 05 で見た `add_conditional_edges` は **不要**。

L187 でやることは:
- 4 つの node を `add_node` で追加
- `START → intent_classifier` の edge
- `qna_agent → compile_followup`、 `refund_agent → compile_followup` の edge
- `compile_followup → END` の edge

</details>

## 参考リンク

- [OpenAI function calling docs](https://platform.openai.com/docs/guides/function-calling) — `tools` 引数の format 詳細
- [OpenAI Chat Completion response object](https://platform.openai.com/docs/api-reference/chat/object) — `tool_calls` の構造、 `finish_reason` の値